# Build Lineage Dependency Graph

This notebook builds a lineage dependency graph from extracted Delta tables.

It focuses on two dependency views:
1. Report visual to semantic model dependencies
2. Semantic model internal dependencies (table, column, measure)

## Inputs (created by extraction notebooks)
- `lineage_reports`
- `lineage_report_visuals`
- `lineage_report_semantic_model_objects`
- `lineage_semantic_models`
- `lineage_semantic_model_columns`
- `lineage_semantic_model_measures`
- `lineage_semantic_model_relationships`

## Outputs
- `lineage_graph_nodes`
- `lineage_graph_edges`
- `lineage_graph_lineage_paths` (optional summarized paths)

The graph outputs are designed for two audiences:
- Developers: impact tracing before model/report changes
- Business users: easier requirement discussions using dependency context

In [ ]:
import re
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple

from pyspark.sql import SparkSession, DataFrame  # type: ignore[reportMissingImports]
from pyspark.sql import functions as F  # type: ignore[reportMissingImports]

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

WRITE_MODE = "overwrite"  # overwrite for deterministic graph rebuild, or append for incremental
BUILD_PATH_SUMMARY = True

REQUIRED_INPUT_TABLES = [
    "lineage_reports",
    "lineage_report_visuals",
    "lineage_report_semantic_model_objects",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_relationships"
]

OUTPUT_TABLE_NODES = "lineage_graph_nodes"
OUTPUT_TABLE_EDGES = "lineage_graph_edges"
OUTPUT_TABLE_PATHS = "lineage_graph_lineage_paths"

print("Graph build configuration ready")
print(f"WRITE_MODE: {WRITE_MODE}")

In [ ]:
def _delta_table_exists(table_name: str) -> bool:
    return bool(spark.catalog.tableExists(table_name))


def _load_delta_table(table_name: str) -> DataFrame:
    return spark.table(table_name)


def _drop_table_if_exists(table_name: str) -> None:
    if _delta_table_exists(table_name):
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")


def _append_to_delta(table_name: str, rows: List[Dict], mode: str = "append") -> int:
    if not rows:
        return 0

    if mode == "overwrite":
        _drop_table_if_exists(table_name)

    df = spark.createDataFrame(rows)
    writer = df.write.format("delta").option("mergeSchema", "true")

    if mode == "overwrite":
        writer.mode("overwrite").saveAsTable(table_name)
    else:
        writer.mode("append").saveAsTable(table_name)

    return len(rows)


def _first_existing_column(df: DataFrame, candidates: List[str]) -> Optional[str]:
    cols = set(df.columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    return None


def _coalesce_string(row: Dict, keys: List[str], default: str = "") -> str:
    for key in keys:
        value = row.get(key)
        if value is not None and str(value).strip() != "":
            return str(value)
    return default


def _normalize_name(value: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9_]+", "_", str(value).strip().lower())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "unknown"


def _make_node_id(entity_type: str, parts: List[str]) -> str:
    normalized_parts = [
        _normalize_name(part)
        for part in parts
        if part is not None and str(part).strip() != ""
    ]
    if not normalized_parts:
        normalized_parts = ["unknown"]
    return f"{entity_type}:{'|'.join(normalized_parts)}"


def _parse_dax_object_refs(expression: str) -> Set[Tuple[str, str]]:
    """
    Best-effort parser for references in DAX-like expressions.

    Returns tuples of (table_name, object_name) where:
    - table_name may be empty for measure-only refs like [Total Sales]
    - object_name is column/measure name
    """
    if not expression:
        return set()

    refs: Set[Tuple[str, str]] = set()

    pattern_table_column = re.compile(r"'([^']+)'\[([^\]]+)\]|([A-Za-z0-9_]+)\[([^\]]+)\]")
    for match in pattern_table_column.finditer(expression):
        quoted_table = match.group(1)
        quoted_obj = match.group(2)
        plain_table = match.group(3)
        plain_obj = match.group(4)

        table_name = quoted_table if quoted_table is not None else plain_table
        object_name = quoted_obj if quoted_obj is not None else plain_obj
        refs.add((str(table_name), str(object_name)))

    pattern_measure_only = re.compile(r"\[([^\]]+)\]")
    for match in pattern_measure_only.finditer(expression):
        measure_name = str(match.group(1))
        refs.add(("", measure_name))

    return refs


print("Helper functions ready")

In [ ]:
# Validate required inputs
missing_inputs = [t for t in REQUIRED_INPUT_TABLES if not _delta_table_exists(t)]
if missing_inputs:
    raise ValueError(f"Missing required input tables: {missing_inputs}")

print("All required input tables found")

reports_df = _load_delta_table("lineage_reports")
visuals_df = _load_delta_table("lineage_report_visuals")
report_objects_df = _load_delta_table("lineage_report_semantic_model_objects")
model_columns_df = _load_delta_table("lineage_semantic_model_columns")
model_measures_df = _load_delta_table("lineage_semantic_model_measures")
model_relationships_df = _load_delta_table("lineage_semantic_model_relationships")

visual_name_col = _first_existing_column(visuals_df, ["visual_name", "name", "title", "object_name"])
visual_id_col = _first_existing_column(visuals_df, ["visual_id", "object_id", "id"])
report_id_col_visuals = _first_existing_column(visuals_df, ["report_id"])

report_id_col_objects = _first_existing_column(report_objects_df, ["report_id"])
object_name_col = _first_existing_column(report_objects_df, ["object_name", "name"])
object_table_col = _first_existing_column(report_objects_df, ["table_name", "table"])
object_type_col = _first_existing_column(report_objects_df, ["object_type", "type"])
visual_name_col_objects = _first_existing_column(report_objects_df, ["visual_name", "visual"])
visual_id_col_objects = _first_existing_column(report_objects_df, ["visual_id", "object_id"])

model_id_col_columns = _first_existing_column(model_columns_df, ["semantic_model_id", "dataset_id", "model_id"])
table_name_col_columns = _first_existing_column(model_columns_df, ["table_name", "table", "table_id"])
column_name_col_columns = _first_existing_column(model_columns_df, ["column_name", "name"])
column_expression_col = _first_existing_column(model_columns_df, ["expression", "formula", "dax_expression"])

model_id_col_measures = _first_existing_column(model_measures_df, ["semantic_model_id", "dataset_id", "model_id"])
table_name_col_measures = _first_existing_column(model_measures_df, ["table_name", "table", "table_id"])
measure_name_col = _first_existing_column(model_measures_df, ["measure_name", "name"])
measure_expression_col = _first_existing_column(model_measures_df, ["expression", "formula", "dax_expression"])

model_id_col_rel = _first_existing_column(model_relationships_df, ["semantic_model_id", "dataset_id", "model_id"])
from_table_col = _first_existing_column(model_relationships_df, ["from_table", "table_from", "from_table_name"])
from_column_col = _first_existing_column(model_relationships_df, ["from_column", "column_from", "from_column_name"])
to_table_col = _first_existing_column(model_relationships_df, ["to_table", "table_to", "to_table_name"])
to_column_col = _first_existing_column(model_relationships_df, ["to_column", "column_to", "to_column_name"])

if not all([report_id_col_objects, object_name_col]):
    raise ValueError("lineage_report_semantic_model_objects is missing required columns (report_id/object_name)")
if not all([table_name_col_columns, column_name_col_columns]):
    raise ValueError("lineage_semantic_model_columns is missing required columns (table_name/column_name)")
if not all([table_name_col_measures, measure_name_col]):
    raise ValueError("lineage_semantic_model_measures is missing required columns (table_name/measure_name)")

# Build lookup for report name and dataset id
report_lookup = {}
for row in reports_df.select("report_id", "report_name", "dataset_id").collect():
    report_lookup[str(row["report_id"])] = {
        "report_name": row["report_name"],
        "dataset_id": row["dataset_id"]
    }

# Build lookup for visuals (report_id + visual_name)
visual_lookup = {}
if visual_name_col and report_id_col_visuals:
    visuals_rows = visuals_df.select(report_id_col_visuals, visual_name_col, *( [visual_id_col] if visual_id_col else [] )).collect()
    for row in visuals_rows:
        report_id = str(row[report_id_col_visuals])
        visual_name = str(row[visual_name_col])
        visual_key = f"{report_id}|{visual_name}"
        visual_lookup[visual_key] = {
            "visual_id": str(row[visual_id_col]) if visual_id_col and row[visual_id_col] is not None else "",
            "visual_name": visual_name
        }

nodes: Dict[str, Dict] = {}
edges: List[Dict] = []

# 1) Nodes and edges for report visual -> semantic object dependencies
report_object_rows = report_objects_df.collect()
for row in report_object_rows:
    row_dict = row.asDict(recursive=True)

    report_id = str(row_dict.get(report_id_col_objects))
    report_name = str(report_lookup.get(report_id, {}).get("report_name") or report_id)
    dataset_id = str(report_lookup.get(report_id, {}).get("dataset_id") or "")

    visual_name = _coalesce_string(row_dict, [visual_name_col_objects or "", "visual_name", "visual"], default="unknown_visual")
    visual_id = _coalesce_string(row_dict, [visual_id_col_objects or "", "visual_id", "object_id"], default="")

    sm_object_name = _coalesce_string(row_dict, [object_name_col or "", "object_name", "name"], default="unknown_object")
    sm_table_name = _coalesce_string(row_dict, [object_table_col or "", "table_name", "table"], default="")
    sm_object_type = _coalesce_string(row_dict, [object_type_col or "", "object_type", "type"], default="unknown")

    visual_node_id = _make_node_id("visual", [dataset_id, report_id, visual_id, visual_name])
    object_node_id = _make_node_id("semantic_object", [dataset_id, sm_table_name, sm_object_name, sm_object_type])
    report_node_id = _make_node_id("report", [dataset_id, report_id, report_name])

    nodes[report_node_id] = {
        "node_id": report_node_id,
        "entity_type": "report",
        "dataset_id": dataset_id,
        "report_id": report_id,
        "display_name": report_name,
        "table_name": None,
        "object_name": report_name,
        "object_subtype": "report"
    }

    nodes[visual_node_id] = {
        "node_id": visual_node_id,
        "entity_type": "visual",
        "dataset_id": dataset_id,
        "report_id": report_id,
        "display_name": visual_name,
        "table_name": None,
        "object_name": visual_name,
        "object_subtype": "visual"
    }

    nodes[object_node_id] = {
        "node_id": object_node_id,
        "entity_type": "semantic_object",
        "dataset_id": dataset_id,
        "report_id": None,
        "display_name": f"{sm_table_name}.{sm_object_name}" if sm_table_name else sm_object_name,
        "table_name": sm_table_name or None,
        "object_name": sm_object_name,
        "object_subtype": sm_object_type
    }

    edges.append({
        "edge_id": _make_node_id("edge", [report_node_id, visual_node_id, "contains_visual"]),
        "from_node_id": report_node_id,
        "to_node_id": visual_node_id,
        "edge_type": "contains_visual",
        "dataset_id": dataset_id,
        "report_id": report_id,
        "evidence": None
    })

    edges.append({
        "edge_id": _make_node_id("edge", [visual_node_id, object_node_id, "uses_semantic_object"]),
        "from_node_id": visual_node_id,
        "to_node_id": object_node_id,
        "edge_type": "uses_semantic_object",
        "dataset_id": dataset_id,
        "report_id": report_id,
        "evidence": None
    })

# 2) Nodes and edges for semantic model internal dependencies
measure_nodes_by_name: Dict[Tuple[str, str], str] = {}
column_nodes_by_table_name: Dict[Tuple[str, str, str], str] = {}

for row in model_columns_df.collect():
    row_dict = row.asDict(recursive=True)

    dataset_id = str(row_dict.get(model_id_col_columns) or "")
    table_name = str(row_dict.get(table_name_col_columns) or "")
    column_name = str(row_dict.get(column_name_col_columns) or "")
    expression = str(row_dict.get(column_expression_col) or "")

    table_node_id = _make_node_id("table", [dataset_id, table_name])
    column_node_id = _make_node_id("column", [dataset_id, table_name, column_name])

    nodes[table_node_id] = {
        "node_id": table_node_id,
        "entity_type": "table",
        "dataset_id": dataset_id,
        "report_id": None,
        "display_name": table_name,
        "table_name": table_name,
        "object_name": table_name,
        "object_subtype": "table"
    }

    nodes[column_node_id] = {
        "node_id": column_node_id,
        "entity_type": "column",
        "dataset_id": dataset_id,
        "report_id": None,
        "display_name": f"{table_name}.{column_name}",
        "table_name": table_name,
        "object_name": column_name,
        "object_subtype": "column"
    }

    column_nodes_by_table_name[(dataset_id, table_name, column_name)] = column_node_id

    edges.append({
        "edge_id": _make_node_id("edge", [table_node_id, column_node_id, "contains_column"]),
        "from_node_id": table_node_id,
        "to_node_id": column_node_id,
        "edge_type": "contains_column",
        "dataset_id": dataset_id,
        "report_id": None,
        "evidence": None
    })

    refs = _parse_dax_object_refs(expression)
    for ref_table, ref_object in refs:
        if ref_table and ref_object:
            source_col_node_id = _make_node_id("column", [dataset_id, ref_table, ref_object])
            edges.append({
                "edge_id": _make_node_id("edge", [source_col_node_id, column_node_id, "column_depends_on"]),
                "from_node_id": source_col_node_id,
                "to_node_id": column_node_id,
                "edge_type": "column_depends_on",
                "dataset_id": dataset_id,
                "report_id": None,
                "evidence": expression[:500]
            })

for row in model_measures_df.collect():
    row_dict = row.asDict(recursive=True)

    dataset_id = str(row_dict.get(model_id_col_measures) or "")
    table_name = str(row_dict.get(table_name_col_measures) or "")
    measure_name = str(row_dict.get(measure_name_col) or "")
    expression = str(row_dict.get(measure_expression_col) or "")

    table_node_id = _make_node_id("table", [dataset_id, table_name])
    measure_node_id = _make_node_id("measure", [dataset_id, table_name, measure_name])

    nodes[table_node_id] = {
        "node_id": table_node_id,
        "entity_type": "table",
        "dataset_id": dataset_id,
        "report_id": None,
        "display_name": table_name,
        "table_name": table_name,
        "object_name": table_name,
        "object_subtype": "table"
    }

    nodes[measure_node_id] = {
        "node_id": measure_node_id,
        "entity_type": "measure",
        "dataset_id": dataset_id,
        "report_id": None,
        "display_name": f"{table_name}.{measure_name}",
        "table_name": table_name,
        "object_name": measure_name,
        "object_subtype": "measure"
    }

    measure_nodes_by_name[(dataset_id, measure_name)] = measure_node_id

    edges.append({
        "edge_id": _make_node_id("edge", [table_node_id, measure_node_id, "contains_measure"]),
        "from_node_id": table_node_id,
        "to_node_id": measure_node_id,
        "edge_type": "contains_measure",
        "dataset_id": dataset_id,
        "report_id": None,
        "evidence": None
    })

    refs = _parse_dax_object_refs(expression)
    for ref_table, ref_object in refs:
        if ref_table and ref_object:
            source_col_node_id = _make_node_id("column", [dataset_id, ref_table, ref_object])
            edges.append({
                "edge_id": _make_node_id("edge", [source_col_node_id, measure_node_id, "measure_depends_on_column"]),
                "from_node_id": source_col_node_id,
                "to_node_id": measure_node_id,
                "edge_type": "measure_depends_on_column",
                "dataset_id": dataset_id,
                "report_id": None,
                "evidence": expression[:500]
            })
        elif ref_object:
            ref_measure_node = measure_nodes_by_name.get((dataset_id, ref_object))
            if ref_measure_node:
                edges.append({
                    "edge_id": _make_node_id("edge", [ref_measure_node, measure_node_id, "measure_depends_on_measure"]),
                    "from_node_id": ref_measure_node,
                    "to_node_id": measure_node_id,
                    "edge_type": "measure_depends_on_measure",
                    "dataset_id": dataset_id,
                    "report_id": None,
                    "evidence": expression[:500]
                })

if all([from_table_col, from_column_col, to_table_col, to_column_col]):
    for row in model_relationships_df.collect():
        row_dict = row.asDict(recursive=True)
        dataset_id = str(row_dict.get(model_id_col_rel) or "")

        from_table = str(row_dict.get(from_table_col) or "")
        from_column = str(row_dict.get(from_column_col) or "")
        to_table = str(row_dict.get(to_table_col) or "")
        to_column = str(row_dict.get(to_column_col) or "")

        from_col_node_id = _make_node_id("column", [dataset_id, from_table, from_column])
        to_col_node_id = _make_node_id("column", [dataset_id, to_table, to_column])

        edges.append({
            "edge_id": _make_node_id("edge", [from_col_node_id, to_col_node_id, "relationship"]),
            "from_node_id": from_col_node_id,
            "to_node_id": to_col_node_id,
            "edge_type": "relationship",
            "dataset_id": dataset_id,
            "report_id": None,
            "evidence": None
        })

# Remove duplicate edges by edge_id
unique_edges: Dict[str, Dict] = {}
for edge in edges:
    unique_edges[edge["edge_id"]] = edge

edge_rows = list(unique_edges.values())
node_rows = list(nodes.values())

print(f"Built {len(node_rows)} graph node(s)")
print(f"Built {len(edge_rows)} graph edge(s)")

nodes_written = _append_to_delta(OUTPUT_TABLE_NODES, node_rows, mode=WRITE_MODE)
edges_written = _append_to_delta(OUTPUT_TABLE_EDGES, edge_rows, mode=WRITE_MODE)

print(f"Wrote {nodes_written} node row(s) to {OUTPUT_TABLE_NODES}")
print(f"Wrote {edges_written} edge row(s) to {OUTPUT_TABLE_EDGES}")

if BUILD_PATH_SUMMARY:
    paths_df = (
        spark.table(OUTPUT_TABLE_EDGES)
        .groupBy("dataset_id", "edge_type")
        .agg(F.count("*").alias("edge_count"))
        .orderBy("dataset_id", "edge_type")
    )

    if WRITE_MODE == "overwrite":
        _drop_table_if_exists(OUTPUT_TABLE_PATHS)
    paths_df.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE_PATHS)
    print(f"Wrote path summary table: {OUTPUT_TABLE_PATHS}")

In [ ]:
def _trace_impacted_nodes(start_node_id: str, max_depth: int = 3) -> DataFrame:
    edges_df = spark.table(OUTPUT_TABLE_EDGES).select("from_node_id", "to_node_id", "edge_type", "dataset_id")

    frontier = spark.createDataFrame([{"node_id": start_node_id, "depth": 0}])
    visited = frontier

    for depth in range(1, max_depth + 1):
        next_hop = (
            frontier.alias("f")
            .join(edges_df.alias("e"), F.col("f.node_id") == F.col("e.from_node_id"), "inner")
            .select(F.col("e.to_node_id").alias("node_id"))
            .distinct()
            .withColumn("depth", F.lit(depth))
        )

        next_hop = next_hop.join(visited.select("node_id"), on="node_id", how="left_anti")
        if next_hop.limit(1).count() == 0:
            break

        visited = visited.unionByName(next_hop)
        frontier = next_hop

    return (
        visited.alias("v")
        .join(spark.table(OUTPUT_TABLE_NODES).alias("n"), F.col("v.node_id") == F.col("n.node_id"), "left")
        .select("v.depth", "n.node_id", "n.entity_type", "n.display_name", "n.dataset_id", "n.object_subtype")
        .orderBy("depth", "entity_type", "display_name")
    )


def find_node_id(entity_type: str, dataset_id: str, table_name: str = "", object_name: str = "") -> str:
    parts = [dataset_id]
    if table_name:
        parts.append(table_name)
    if object_name:
        parts.append(object_name)
    return _make_node_id(entity_type, parts)


print("Impact tracing helpers ready")

# Example usage:
# sample_node_id = find_node_id("measure", dataset_id="<dataset_id>", table_name="Sales", object_name="Total Sales")
# display(_trace_impacted_nodes(sample_node_id, max_depth=4))